# 🥊 Search Comparison: Dense vs ColBERT
## Head-to-head battle between traditional RAG and late interaction!

This notebook provides comprehensive comparisons between **Dense Retrieval** (notebook 2) and **ColBERT** (notebook 3) across multiple dimensions:

### 🎯 What We'll Compare:
1. **Search Quality**: Relevance and precision of results
2. **Performance**: Speed and computational efficiency
3. **Complex Query Handling**: Multi-constraint and contextual searches
4. **Token-Level Analysis**: Understanding WHY results differ
5. **Real-World Scenarios**: Restaurant discovery use cases
6. **Scalability**: Storage and computational trade-offs

### 🏆 Expected Outcomes:
- **Dense**: Fast, simple, good for basic similarity
- **ColBERT**: More nuanced, better for complex queries, interpretable

Let's find out which approach truly wins for restaurant search! 🍽️

In [ ]:
# Setup environment and imports
import sys
sys.path.append('../..')  # Add project root to path
from setup import *

# Additional imports for comparison
import time
import json
from typing import List, Dict, Tuple, Any
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

print("✅ Setup imported successfully!")
print(f"📂 Working directory: {os.getcwd()}")
print(f"🎯 Project root: {os.getenv('PROJECT_ROOT')}")
print(f"🔧 Device: {get_device()}")

In [ ]:
# Load dense retrieval setup
from sentence_transformers import SentenceTransformer
import lancedb

print("🔄 Loading Dense Retrieval Setup...")

# Load dense model
dense_model_name = os.getenv('DENSE_MODEL_NAME')
device_for_dense = 'cpu' if get_device() == 'mps' else get_device()
dense_model = SentenceTransformer(dense_model_name, device=device_for_dense)

# Connect to vector database
db_path = os.getenv('VECTOR_STORE_DIR')
db = lancedb.connect(db_path)

# Load dense table
dense_table_name = "dense_reviews"
if dense_table_name in db.table_names():
    dense_table = db.open_table(dense_table_name)
    print(f"✅ Dense table loaded: {len(dense_table)} records")
else:
    print(f"❌ Dense table not found. Please run notebook 2 first.")
    dense_table = None

print(f"📦 Dense model: {dense_model_name}")
print(f"🖥️  Dense device: {device_for_dense}")

In [ ]:
# Load ColBERT setup
from pylate import models
import torch

print("🔄 Loading ColBERT Setup...")

# Load ColBERT model
colbert_model_name = os.getenv('COLBERT_MODEL_NAME')
device_for_colbert = 'cpu'  # ColBERT works better on CPU for M1

colbert_model = models.ColBERT(
    model_name_or_path=colbert_model_name,
    device=device_for_colbert
)

# Load ColBERT table
colbert_table_name = "colbert_restaurant_reviews"
if colbert_table_name in db.table_names():
    colbert_table = db.open_table(colbert_table_name)
    print(f"✅ ColBERT table loaded: {len(colbert_table)} records")
else:
    print(f"❌ ColBERT table not found. Please run notebook 3 first.")
    colbert_table = None

print(f"🔍 ColBERT model: {colbert_model_name}")
print(f"🖥️  ColBERT device: {device_for_colbert}")

# Load original data for reference
data_path = os.getenv('RESTAURANT_REVIEWS_CSV')
df = pd.read_csv(data_path)
print(f"📊 Original data: {len(df)} restaurant reviews")

## 📊 Comparison Framework

Let's define our comparison framework to systematically evaluate both approaches.

In [ ]:
@dataclass
class SearchResult:
    """Standardized search result for comparison"""
    id: int
    restaurant: str
    review: str
    rating: int
    reviewer: str
    score: float
    method: str  # 'dense' or 'colbert'
    search_time: float
    
@dataclass
class ComparisonMetrics:
    """Metrics for comparing search methods"""
    query: str
    dense_results: List[SearchResult]
    colbert_results: List[SearchResult]
    dense_search_time: float
    colbert_search_time: float
    overlap_percentage: float
    avg_rating_diff: float

def dense_search(query: str, top_k: int = 3) -> Tuple[List[SearchResult], float]:
    """Perform dense retrieval search"""
    if dense_table is None:
        return [], 0.0
        
    start_time = time.time()
    
    # Encode query
    query_embedding = dense_model.encode(query)
    
    # Search
    results = dense_table.search(query_embedding).limit(top_k).to_pandas()
    
    search_time = time.time() - start_time
    
    # Convert to SearchResult objects
    search_results = []
    for _, row in results.iterrows():
        distance = row.get('_distance', 0)
        similarity = 1 - distance  # Convert distance to similarity
        
        result = SearchResult(
            id=int(row['id']),
            restaurant=row['restaurant'],
            review=row['review'],
            rating=int(row['rating']),
            reviewer=row['reviewer'],
            score=similarity,
            method='dense',
            search_time=search_time
        )
        search_results.append(result)
    
    return search_results, search_time

def colbert_search(query: str, top_k: int = 3) -> Tuple[List[SearchResult], float]:
    """Perform ColBERT search with MaxSim"""
    if colbert_table is None:
        return [], 0.0
        
    start_time = time.time()
    
    # Encode query with ColBERT
    query_embeddings = colbert_model.encode([query], is_query=True)[0]
    
    # Get all documents
    all_docs = colbert_table.to_pandas()
    
    results = []
    
    # Compute MaxSim for each document
    for _, row in all_docs.iterrows():
        doc_embeddings = torch.tensor(row['embeddings'], dtype=torch.float32)
        
        # Compute similarity matrix
        similarity_matrix = torch.matmul(query_embeddings, doc_embeddings.T)
        
        # MaxSim operation
        max_similarities = torch.max(similarity_matrix, dim=1)[0]
        score = torch.sum(max_similarities).item()
        
        result = SearchResult(
            id=int(row['id']),
            restaurant=row['restaurant'],
            review=row['review'],
            rating=int(row['rating']),
            reviewer=row['reviewer'],
            score=score,
            method='colbert',
            search_time=0.0  # Will be set below
        )
        results.append(result)
    
    # Sort by score and take top_k
    results.sort(key=lambda x: x.score, reverse=True)
    top_results = results[:top_k]
    
    search_time = time.time() - start_time
    
    # Update search time for all results
    for result in top_results:
        result.search_time = search_time
    
    return top_results, search_time

print("✅ Search functions defined!")
print("🎯 Ready for head-to-head comparisons!")

## 🔍 Test Queries for Comparison

Let's define a diverse set of queries that showcase different search scenarios.

In [ ]:
# Define test queries with expected strengths
test_queries = [
    {
        'query': 'Italian authentic pasta',
        'description': 'Simple keyword search - should be comparable',
        'expected_winner': 'tie'
    },
    {
        'query': 'good for working laptop wifi quiet',
        'description': 'Multi-constraint query - ColBERT should excel',
        'expected_winner': 'colbert'
    },
    {
        'query': 'expensive fine dining special occasion worth money',
        'description': 'Complex contextual query - ColBERT advantage',
        'expected_winner': 'colbert'
    },
    {
        'query': 'family friendly kids children large groups',
        'description': 'Synonym-rich query - both should handle well',
        'expected_winner': 'tie'
    },
    {
        'query': 'romantic date night intimate atmosphere candles',
        'description': 'Emotional/contextual search - ColBERT strength',
        'expected_winner': 'colbert'
    },
    {
        'query': 'budget cheap affordable inexpensive student',
        'description': 'Multiple synonyms - dense might struggle',
        'expected_winner': 'colbert'
    },
    {
        'query': 'outdoor seating patio garden terrace',
        'description': 'Specific feature with variants',
        'expected_winner': 'colbert'
    },
    {
        'query': 'disappointing overpriced terrible service slow',
        'description': 'Negative sentiment query',
        'expected_winner': 'colbert'
    }
]

print(f"📋 Defined {len(test_queries)} test queries:")
for i, test in enumerate(test_queries, 1):
    print(f"   {i}. '{test['query']}'")
    print(f"      {test['description']}")
    print(f"      Expected: {test['expected_winner']}")
    print()

## ⚡ Performance Comparison

First, let's compare the raw performance metrics - speed and computational efficiency.

In [ ]:
def run_performance_comparison(query: str, num_runs: int = 3) -> Dict:
    """Run performance comparison for a single query"""
    print(f"⚡ Performance test: '{query}'")
    
    # Run multiple times for accurate timing
    dense_times = []
    colbert_times = []
    
    for i in range(num_runs):
        # Dense search
        _, dense_time = dense_search(query, top_k=3)
        dense_times.append(dense_time)
        
        # ColBERT search  
        _, colbert_time = colbert_search(query, top_k=3)
        colbert_times.append(colbert_time)
    
    # Calculate statistics
    avg_dense_time = np.mean(dense_times)
    avg_colbert_time = np.mean(colbert_times)
    
    speedup_ratio = avg_colbert_time / avg_dense_time if avg_dense_time > 0 else 0
    
    results = {
        'query': query,
        'dense_avg_time': avg_dense_time,
        'colbert_avg_time': avg_colbert_time,
        'dense_std': np.std(dense_times),
        'colbert_std': np.std(colbert_times),
        'speedup_ratio': speedup_ratio,
        'faster_method': 'dense' if avg_dense_time < avg_colbert_time else 'colbert'
    }
    
    print(f"   Dense:   {avg_dense_time:.4f}s (±{np.std(dense_times):.4f})")
    print(f"   ColBERT: {avg_colbert_time:.4f}s (±{np.std(colbert_times):.4f})")
    print(f"   Ratio:   {speedup_ratio:.1f}x {'slower' if speedup_ratio > 1 else 'faster'} (ColBERT vs Dense)")
    
    return results

# Run performance comparison on a subset of queries
performance_results = []
print("🚀 PERFORMANCE COMPARISON")
print("=" * 50)

for test_query in test_queries[:4]:  # Test first 4 queries
    result = run_performance_comparison(test_query['query'])
    performance_results.append(result)
    print()

# Summary statistics
avg_dense_time = np.mean([r['dense_avg_time'] for r in performance_results])
avg_colbert_time = np.mean([r['colbert_avg_time'] for r in performance_results])
overall_ratio = avg_colbert_time / avg_dense_time

print(f"📊 PERFORMANCE SUMMARY:")
print(f"   Average Dense time:   {avg_dense_time:.4f}s")
print(f"   Average ColBERT time: {avg_colbert_time:.4f}s")
print(f"   Overall ratio: {overall_ratio:.1f}x")
print(f"   Winner: {'Dense' if avg_dense_time < avg_colbert_time else 'ColBERT'} for speed")

In [ ]:
# Visualize performance comparison
plt.figure(figsize=(12, 8))

# Performance comparison chart
plt.subplot(2, 2, 1)
queries_short = [r['query'][:20] + '...' if len(r['query']) > 20 else r['query'] for r in performance_results]
dense_times = [r['dense_avg_time'] for r in performance_results]
colbert_times = [r['colbert_avg_time'] for r in performance_results]

x = np.arange(len(queries_short))
width = 0.35

plt.bar(x - width/2, dense_times, width, label='Dense', alpha=0.7, color='lightblue')
plt.bar(x + width/2, colbert_times, width, label='ColBERT', alpha=0.7, color='lightcoral')

plt.xlabel('Queries')
plt.ylabel('Search Time (seconds)')
plt.title('Search Speed Comparison')
plt.xticks(x, queries_short, rotation=45, ha='right')
plt.legend()
plt.grid(True, alpha=0.3)

# Speed ratio visualization
plt.subplot(2, 2, 2)
ratios = [r['speedup_ratio'] for r in performance_results]
colors = ['green' if r < 1 else 'red' for r in ratios]

plt.bar(queries_short, ratios, color=colors, alpha=0.7)
plt.axhline(y=1, color='black', linestyle='--', alpha=0.5)
plt.xlabel('Queries')
plt.ylabel('Speed Ratio (ColBERT/Dense)')
plt.title('Speed Ratio (>1 = ColBERT slower)')
plt.xticks(rotation=45, ha='right')
plt.grid(True, alpha=0.3)

# Memory usage comparison (theoretical)
plt.subplot(2, 2, 3)
methods = ['Dense\nEmbedding', 'ColBERT\nEmbedding']
memory_usage = [0.12, 1.2]  # Example values in MB
colors = ['lightblue', 'lightcoral']

bars = plt.bar(methods, memory_usage, color=colors, alpha=0.7)
plt.ylabel('Memory Usage (MB)')
plt.title('Storage Requirements')
plt.grid(True, alpha=0.3)

for bar, usage in zip(bars, memory_usage):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{usage:.1f} MB', ha='center', va='bottom')

# Computational complexity
plt.subplot(2, 2, 4)
operations = ['Query\nEncoding', 'Similarity\nComputation', 'Result\nRanking']
dense_ops = [1, 1, 1]  # Normalized complexity
colbert_ops = [1, 3, 2]  # Higher due to MaxSim

x = np.arange(len(operations))
width = 0.35

plt.bar(x - width/2, dense_ops, width, label='Dense', alpha=0.7, color='lightblue')
plt.bar(x + width/2, colbert_ops, width, label='ColBERT', alpha=0.7, color='lightcoral')

plt.xlabel('Operation')
plt.ylabel('Relative Complexity')
plt.title('Computational Complexity')
plt.xticks(x, operations)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("📈 Performance visualization complete!")

## 🎯 Search Quality Comparison

Now let's compare the actual search results and see where each method excels.

In [ ]:
def compare_search_results(query: str, top_k: int = 3) -> ComparisonMetrics:
    """Compare search results between dense and ColBERT"""
    print(f"🔍 Comparing results for: '{query}'")
    print("=" * 60)
    
    # Get results from both methods
    dense_results, dense_time = dense_search(query, top_k)
    colbert_results, colbert_time = colbert_search(query, top_k)
    
    # Calculate overlap
    dense_ids = set(r.id for r in dense_results)
    colbert_ids = set(r.id for r in colbert_results)
    overlap = len(dense_ids & colbert_ids)
    overlap_percentage = (overlap / top_k) * 100
    
    # Compare average ratings
    dense_avg_rating = np.mean([r.rating for r in dense_results]) if dense_results else 0
    colbert_avg_rating = np.mean([r.rating for r in colbert_results]) if colbert_results else 0
    avg_rating_diff = colbert_avg_rating - dense_avg_rating
    
    # Display results side by side
    print(f"🚀 Performance: Dense {dense_time:.3f}s vs ColBERT {colbert_time:.3f}s")
    print(f"🎯 Overlap: {overlap}/{top_k} results ({overlap_percentage:.0f}%)")
    print(f"⭐ Avg ratings: Dense {dense_avg_rating:.1f} vs ColBERT {colbert_avg_rating:.1f}")
    print()
    
    # Show results side by side
    for i in range(max(len(dense_results), len(colbert_results))):
        print(f"Rank {i+1}:")
        
        # Dense result
        if i < len(dense_results):
            dr = dense_results[i]
            print(f"   📦 Dense:   {dr.restaurant} (⭐{dr.rating}) Score: {dr.score:.3f}")
            print(f"              {dr.review[:80]}...")
        else:
            print(f"   📦 Dense:   (no result)")
        
        # ColBERT result
        if i < len(colbert_results):
            cr = colbert_results[i]
            print(f"   🔍 ColBERT: {cr.restaurant} (⭐{cr.rating}) Score: {cr.score:.3f}")
            print(f"              {cr.review[:80]}...")
        else:
            print(f"   🔍 ColBERT: (no result)")
        
        # Highlight if same restaurant
        if (i < len(dense_results) and i < len(colbert_results) and 
            dense_results[i].id == colbert_results[i].id):
            print(f"              ✅ SAME RESTAURANT")
        elif (i < len(dense_results) and i < len(colbert_results)):
            print(f"              🔄 DIFFERENT RESTAURANTS")
        print()
    
    return ComparisonMetrics(
        query=query,
        dense_results=dense_results,
        colbert_results=colbert_results,
        dense_search_time=dense_time,
        colbert_search_time=colbert_time,
        overlap_percentage=overlap_percentage,
        avg_rating_diff=avg_rating_diff
    )

print("✅ Search comparison function ready!")

### 🥊 Round 1: Simple Keyword Search

In [ ]:
# Test 1: Simple keyword search
round1_query = "Italian authentic pasta"
round1_results = compare_search_results(round1_query)

print(f"🏆 Round 1 Analysis:")
if round1_results.overlap_percentage > 66:
    print(f"   Result: TIE - High overlap ({round1_results.overlap_percentage:.0f}%)")
elif round1_results.avg_rating_diff > 0.5:
    print(f"   Result: ColBERT WINS - Higher avg rating (+{round1_results.avg_rating_diff:.1f})")
elif round1_results.avg_rating_diff < -0.5:
    print(f"   Result: Dense WINS - Higher avg rating (+{-round1_results.avg_rating_diff:.1f})")
else:
    print(f"   Result: TIE - Similar ratings")

### 🥊 Round 2: Multi-Constraint Query

In [ ]:
# Test 2: Multi-constraint query (ColBERT should excel)
round2_query = "good for working laptop wifi quiet"
round2_results = compare_search_results(round2_query)

print(f"🏆 Round 2 Analysis:")
print(f"   This query has multiple constraints: 'working', 'laptop', 'wifi', 'quiet'")
print(f"   ColBERT can match each constraint separately")
print(f"   Dense must compress all constraints into one similarity score")

if round2_results.overlap_percentage < 34:
    print(f"   Result: DIFFERENT APPROACHES - Low overlap ({round2_results.overlap_percentage:.0f}%)")
    print(f"   This showcases the fundamental difference between the methods")
else:
    print(f"   Result: Similar results - Both methods handled multi-constraints well")

### 🥊 Round 3: Complex Contextual Query

In [ ]:
# Test 3: Complex contextual query
round3_query = "expensive fine dining special occasion worth money"
round3_results = compare_search_results(round3_query)

print(f"🏆 Round 3 Analysis:")
print(f"   Complex query with context: 'expensive' + 'worth money' paradox")
print(f"   ColBERT can understand: expensive BUT worth it")
print(f"   Dense might struggle with contradictory terms")

# Analyze the specific results
for method, results in [('Dense', round3_results.dense_results), ('ColBERT', round3_results.colbert_results)]:
    if results:
        high_rating_count = sum(1 for r in results if r.rating >= 4)
        print(f"   {method}: {high_rating_count}/{len(results)} results are 4+ stars")

### 🥊 Round 4: Negative Sentiment Query

In [ ]:
# Test 4: Negative sentiment query
round4_query = "disappointing overpriced terrible service slow"
round4_results = compare_search_results(round4_query)

print(f"🏆 Round 4 Analysis:")
print(f"   Negative sentiment query - looking for bad reviews")
print(f"   Should return low-rated restaurants")

# Check if results match the sentiment
for method, results in [('Dense', round4_results.dense_results), ('ColBERT', round4_results.colbert_results)]:
    if results:
        low_rating_count = sum(1 for r in results if r.rating <= 2)
        avg_rating = np.mean([r.rating for r in results])
        print(f"   {method}: {low_rating_count}/{len(results)} low ratings, avg: {avg_rating:.1f}")
        
        # Check if reviews contain negative words
        negative_words = ['disappointing', 'overpriced', 'terrible', 'slow', 'bad', 'awful', 'worst']
        for i, result in enumerate(results):
            neg_word_count = sum(1 for word in negative_words if word.lower() in result.review.lower())
            if neg_word_count > 0:
                print(f"     Rank {i+1}: Contains {neg_word_count} negative words")

## 📊 Comprehensive Analysis Dashboard

In [ ]:
# Run all test queries and collect comprehensive metrics
all_comparisons = []
print("🔍 Running comprehensive comparison across all test queries...")
print()

for i, test_config in enumerate(test_queries[:6], 1):  # Test first 6 queries
    print(f"Test {i}/{len(test_queries[:6])}: {test_config['description']}")
    comparison = compare_search_results(test_config['query'], top_k=3)
    all_comparisons.append(comparison)
    print("\n" + "="*80 + "\n")

print("✅ All comparisons complete!")

In [ ]:
# Create comprehensive analysis dashboard
plt.figure(figsize=(16, 12))

# 1. Search Time Comparison
plt.subplot(3, 3, 1)
queries_short = [comp.query[:15] + '...' for comp in all_comparisons]
dense_times = [comp.dense_search_time for comp in all_comparisons]
colbert_times = [comp.colbert_search_time for comp in all_comparisons]

x = np.arange(len(queries_short))
width = 0.35

plt.bar(x - width/2, dense_times, width, label='Dense', alpha=0.8, color='lightblue')
plt.bar(x + width/2, colbert_times, width, label='ColBERT', alpha=0.8, color='lightcoral')

plt.xlabel('Queries')
plt.ylabel('Time (seconds)')
plt.title('Search Speed Comparison')
plt.xticks(x, queries_short, rotation=45, ha='right')
plt.legend()
plt.grid(True, alpha=0.3)

# 2. Result Overlap Analysis
plt.subplot(3, 3, 2)
overlaps = [comp.overlap_percentage for comp in all_comparisons]
colors = ['green' if o >= 66 else 'orange' if o >= 33 else 'red' for o in overlaps]

plt.bar(queries_short, overlaps, color=colors, alpha=0.7)
plt.axhline(y=66, color='green', linestyle='--', alpha=0.5, label='High overlap')
plt.axhline(y=33, color='orange', linestyle='--', alpha=0.5, label='Medium overlap')
plt.xlabel('Queries')
plt.ylabel('Overlap Percentage')
plt.title('Result Overlap Between Methods')
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.grid(True, alpha=0.3)

# 3. Average Rating Comparison
plt.subplot(3, 3, 3)
dense_avg_ratings = [np.mean([r.rating for r in comp.dense_results]) if comp.dense_results else 0 
                    for comp in all_comparisons]
colbert_avg_ratings = [np.mean([r.rating for r in comp.colbert_results]) if comp.colbert_results else 0 
                      for comp in all_comparisons]

plt.scatter(dense_avg_ratings, colbert_avg_ratings, alpha=0.7, s=100, color='purple')
plt.plot([1, 5], [1, 5], 'k--', alpha=0.5, label='Equal ratings')
plt.xlabel('Dense Avg Rating')
plt.ylabel('ColBERT Avg Rating')
plt.title('Result Quality: Average Ratings')
plt.legend()
plt.grid(True, alpha=0.3)

# 4. Speed Ratio Distribution
plt.subplot(3, 3, 4)
speed_ratios = [comp.colbert_search_time / comp.dense_search_time if comp.dense_search_time > 0 else 0 
               for comp in all_comparisons]
plt.hist(speed_ratios, bins=8, alpha=0.7, color='skyblue', edgecolor='black')
plt.axvline(x=1, color='red', linestyle='--', label='Equal speed')
plt.xlabel('Speed Ratio (ColBERT/Dense)')
plt.ylabel('Frequency')
plt.title('Speed Ratio Distribution')
plt.legend()
plt.grid(True, alpha=0.3)

# 5. Query Complexity vs Overlap
plt.subplot(3, 3, 5)
query_complexities = [len(comp.query.split()) for comp in all_comparisons]
plt.scatter(query_complexities, overlaps, alpha=0.7, s=100, color='green')
plt.xlabel('Query Length (words)')
plt.ylabel('Result Overlap %')
plt.title('Query Complexity vs Agreement')
plt.grid(True, alpha=0.3)

# Add trend line
z = np.polyfit(query_complexities, overlaps, 1)
p = np.poly1d(z)
plt.plot(query_complexities, p(query_complexities), "r--", alpha=0.8)

# 6. Method Preference by Query Type
plt.subplot(3, 3, 6)
dense_wins = sum(1 for comp in all_comparisons if np.mean([r.rating for r in comp.dense_results]) > 
                np.mean([r.rating for r in comp.colbert_results]) if comp.dense_results and comp.colbert_results)
colbert_wins = sum(1 for comp in all_comparisons if np.mean([r.rating for r in comp.colbert_results]) > 
                  np.mean([r.rating for r in comp.dense_results]) if comp.dense_results and comp.colbert_results)
ties = len(all_comparisons) - dense_wins - colbert_wins

methods = ['Dense\nWins', 'ColBERT\nWins', 'Ties']
wins = [dense_wins, colbert_wins, ties]
colors = ['lightblue', 'lightcoral', 'lightgray']

plt.pie(wins, labels=methods, colors=colors, autopct='%1.0f%%', startangle=90)
plt.title('Quality Winner Distribution\n(by avg rating)')

# 7. Score Distribution Comparison
plt.subplot(3, 3, 7)
all_dense_scores = []
all_colbert_scores = []

for comp in all_comparisons:
    all_dense_scores.extend([r.score for r in comp.dense_results])
    # Normalize ColBERT scores to 0-1 range for comparison
    colbert_scores = [r.score for r in comp.colbert_results]
    if colbert_scores:
        max_score = max(colbert_scores)
        min_score = min(colbert_scores)
        if max_score > min_score:
            normalized_scores = [(s - min_score) / (max_score - min_score) for s in colbert_scores]
        else:
            normalized_scores = [0.5] * len(colbert_scores)  # All same score
        all_colbert_scores.extend(normalized_scores)

plt.hist(all_dense_scores, bins=10, alpha=0.5, label='Dense', color='lightblue')
plt.hist(all_colbert_scores, bins=10, alpha=0.5, label='ColBERT (norm)', color='lightcoral')
plt.xlabel('Normalized Score')
plt.ylabel('Frequency')
plt.title('Score Distribution Comparison')
plt.legend()
plt.grid(True, alpha=0.3)

# 8. Search Efficiency Metrics
plt.subplot(3, 3, 8)
metrics = ['Avg Search\nTime (ms)', 'Storage\nSize (MB)', 'Query\nComplexity']
dense_metrics = [np.mean(dense_times) * 1000, 0.12, 1]  # Normalized values
colbert_metrics = [np.mean(colbert_times) * 1000, 1.2, 3]  # Higher complexity

x = np.arange(len(metrics))
width = 0.35

plt.bar(x - width/2, dense_metrics, width, label='Dense', alpha=0.8, color='lightblue')
plt.bar(x + width/2, colbert_metrics, width, label='ColBERT', alpha=0.8, color='lightcoral')

plt.xlabel('Metrics')
plt.ylabel('Relative Value')
plt.title('Efficiency Comparison')
plt.xticks(x, metrics)
plt.legend()
plt.grid(True, alpha=0.3)

# 9. Summary Statistics
plt.subplot(3, 3, 9)
plt.axis('off')

# Calculate summary stats
avg_overlap = np.mean(overlaps)
avg_speed_ratio = np.mean(speed_ratios)
avg_dense_rating = np.mean(dense_avg_ratings)
avg_colbert_rating = np.mean(colbert_avg_ratings)

summary_text = f"""
📊 COMPARISON SUMMARY
{'='*25}

🔍 Queries tested: {len(all_comparisons)}
📈 Avg result overlap: {avg_overlap:.1f}%
⚡ Speed ratio: {avg_speed_ratio:.1f}x
⭐ Dense avg rating: {avg_dense_rating:.2f}
⭐ ColBERT avg rating: {avg_colbert_rating:.2f}

🏆 WINNER ANALYSIS:
• Speed: {'Dense' if avg_speed_ratio > 1 else 'ColBERT'}
• Quality: {'ColBERT' if avg_colbert_rating > avg_dense_rating else 'Dense' if avg_dense_rating > avg_colbert_rating else 'Tie'}
• Consistency: {'High' if avg_overlap > 66 else 'Medium' if avg_overlap > 33 else 'Low'}
"""

plt.text(0.05, 0.95, summary_text, transform=plt.gca().transAxes, 
         fontsize=10, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))

plt.tight_layout()
plt.show()

print("📊 Comprehensive analysis dashboard complete!")

## 🔬 Token-Level Analysis: Understanding WHY Results Differ

Let's dive deep into ColBERT's token-level matching to understand what makes it different.

In [ ]:
def analyze_token_matching(query: str, document_id: int):
    """Analyze token-level matching for ColBERT"""
    print(f"🔬 Token-level analysis for query: '{query}'")
    print(f"📄 Document ID: {document_id}")
    
    # Get the specific document
    doc_row = colbert_table.to_pandas()
    doc_row = doc_row[doc_row['id'] == document_id].iloc[0]
    
    print(f"🏪 Restaurant: {doc_row['restaurant']}")
    print(f"⭐ Rating: {doc_row['rating']}/5")
    print(f"📝 Review: {doc_row['review'][:150]}...")
    print()
    
    # Encode query and document
    query_embeddings = colbert_model.encode([query], is_query=True)[0]
    doc_embeddings = torch.tensor(doc_row['embeddings'], dtype=torch.float32)
    
    print(f"🎯 Query tokens: {query_embeddings.shape[0]}")
    print(f"📄 Document tokens: {doc_embeddings.shape[0]}")
    
    # Compute similarity matrix
    similarity_matrix = torch.matmul(query_embeddings, doc_embeddings.T)
    
    # Find best matches for each query token
    query_words = query.split()
    doc_words = doc_row['review'].split()
    
    print(f"\n🔍 Token Matching Analysis:")
    print("-" * 50)
    
    for i in range(min(len(query_words), query_embeddings.shape[0])):
        # Find best matching document token for this query token
        best_doc_token_idx = torch.argmax(similarity_matrix[i]).item()
        best_similarity = similarity_matrix[i][best_doc_token_idx].item()
        
        # Map back to approximate words (this is simplified)
        query_word = query_words[min(i, len(query_words)-1)]
        doc_token_approx_idx = min(best_doc_token_idx, len(doc_words)-1)
        doc_word = doc_words[doc_token_approx_idx]
        
        print(f"   '{query_word}' → '{doc_word}' (similarity: {best_similarity:.3f})")
    
    # Calculate overall MaxSim score
    max_similarities = torch.max(similarity_matrix, dim=1)[0]
    total_score = torch.sum(max_similarities).item()
    
    print(f"\n📊 MaxSim breakdown:")
    for i, sim in enumerate(max_similarities[:len(query_words)]):
        query_word = query_words[min(i, len(query_words)-1)]
        print(f"   '{query_word}': {sim.item():.3f}")
    
    print(f"\n🎯 Total ColBERT score: {total_score:.3f}")
    
    return similarity_matrix, total_score

# Analyze a specific example
analysis_query = "romantic date night intimate"
analysis_doc_id = 1  # Choose a specific restaurant

sim_matrix, score = analyze_token_matching(analysis_query, analysis_doc_id)

In [ ]:
# Visualize the similarity matrix
plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
plt.imshow(sim_matrix.numpy(), cmap='RdYlBu_r', aspect='auto')
plt.colorbar(label='Similarity Score')
plt.xlabel('Document Tokens')
plt.ylabel('Query Tokens')
plt.title('Token-to-Token Similarity Matrix\n(ColBERT Magic!)')

# Show MaxSim operation
plt.subplot(2, 2, 2)
max_sims = torch.max(sim_matrix, dim=1)[0].numpy()
query_words = analysis_query.split()

plt.bar(range(len(max_sims)), max_sims, alpha=0.7, color='coral')
plt.xlabel('Query Tokens')
plt.ylabel('Max Similarity')
plt.title('MaxSim per Query Token')
plt.xticks(range(len(query_words)), query_words, rotation=45)
plt.grid(True, alpha=0.3)

# Compare with theoretical dense approach
plt.subplot(2, 2, 3)
methods = ['Dense\n(Single Score)', 'ColBERT\n(MaxSim)']
# Simulate dense score as average similarity
dense_sim_approx = sim_matrix.mean().item()
colbert_score = score

scores = [dense_sim_approx, colbert_score]
colors = ['lightblue', 'lightcoral']

bars = plt.bar(methods, scores, color=colors, alpha=0.7)
plt.ylabel('Similarity Score')
plt.title('Dense vs ColBERT Scoring')
plt.grid(True, alpha=0.3)

for bar, score_val in zip(bars, scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{score_val:.2f}', ha='center', va='bottom')

# Information preservation comparison
plt.subplot(2, 2, 4)
plt.axis('off')

explanation_text = f"""
🔍 WHY COLBERT IS DIFFERENT:

Dense Retrieval:
• Query → Single 384-dim vector
• Document → Single 384-dim vector  
• Similarity = dot product
• Result: One similarity score

ColBERT:
• Query → {sim_matrix.shape[0]} token vectors
• Document → {sim_matrix.shape[1]} token vectors
• Similarity = MaxSim operation
• Result: Fine-grained matching

🎯 ColBERT finds the BEST match for
each query term, preserving nuance!
"""

plt.text(0.05, 0.95, explanation_text, transform=plt.gca().transAxes, 
         fontsize=10, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

print("🔬 Token-level analysis visualization complete!")

## 🌟 Real-World Restaurant Discovery Scenarios

Let's test both methods on realistic restaurant discovery scenarios to see practical differences.

In [ ]:
# Define realistic restaurant discovery scenarios
scenarios = [
    {
        'name': '🏢 Business Lunch',
        'query': 'quiet business lunch meeting professional atmosphere',
        'context': 'Need a place for important business discussion'
    },
    {
        'name': '👨‍👩‍👧‍👦 Family Dinner',
        'query': 'family friendly kids children large portions sharing',
        'context': 'Weekend dinner with kids and grandparents'
    },
    {
        'name': '💑 Date Night',
        'query': 'romantic intimate candlelight special occasion anniversary',
        'context': 'Celebrating wedding anniversary'
    },
    {
        'name': '🎓 Student Budget',
        'query': 'cheap affordable student budget large portions value',
        'context': 'College student looking for good value'
    },
    {
        'name': '🌱 Health Conscious',
        'query': 'healthy fresh organic vegetarian light options salad',
        'context': 'Following a healthy diet plan'
    }
]

print("🌟 REAL-WORLD RESTAURANT DISCOVERY SCENARIOS")
print("=" * 60)

scenario_results = []

for scenario in scenarios:
    print(f"\n{scenario['name']} Scenario")
    print(f"Context: {scenario['context']}")
    print(f"Query: '{scenario['query']}'")
    print("-" * 40)
    
    # Get results from both methods
    dense_results, dense_time = dense_search(scenario['query'], top_k=2)
    colbert_results, colbert_time = colbert_search(scenario['query'], top_k=2)
    
    print(f"📦 Dense Results ({dense_time:.3f}s):")
    for i, result in enumerate(dense_results, 1):
        print(f"   {i}. {result.restaurant} (⭐{result.rating}) - Score: {result.score:.3f}")
        print(f"      {result.review[:100]}...")
    
    print(f"\n🔍 ColBERT Results ({colbert_time:.3f}s):")
    for i, result in enumerate(colbert_results, 1):
        print(f"   {i}. {result.restaurant} (⭐{result.rating}) - Score: {result.score:.3f}")
        print(f"      {result.review[:100]}...")
    
    # Analyze which is more suitable for the scenario
    scenario_analysis = {
        'scenario': scenario['name'],
        'query': scenario['query'],
        'dense_results': dense_results,
        'colbert_results': colbert_results,
        'dense_time': dense_time,
        'colbert_time': colbert_time
    }
    scenario_results.append(scenario_analysis)
    
    print(f"\n🤔 Which is better for this scenario?")
    if scenario['name'] in ['🏢 Business Lunch', '💑 Date Night', '🌱 Health Conscious']:
        print(f"   Expected: ColBERT (complex contextual matching)")
    else:
        print(f"   Expected: Both should work well")
    
    print("\n" + "="*80)

## 📋 Final Scorecard & Recommendations

Let's create a final scorecard comparing both methods across all dimensions.

In [ ]:
# Create comprehensive scorecard
scorecard_data = {
    'Criteria': [
        'Search Speed ⚡',
        'Storage Efficiency 💾',
        'Simple Queries 🔍',
        'Complex Queries 🧩',
        'Multi-Constraint 🎯',
        'Contextual Understanding 🧠',
        'Interpretability 🔬',
        'Implementation Simplicity 🛠️',
        'Scalability 📈',
        'Research Backing 📚'
    ],
    'Dense Retrieval': [9, 10, 8, 6, 5, 6, 4, 10, 9, 8],
    'ColBERT': [7, 6, 8, 9, 10, 9, 10, 7, 7, 9],
    'Description': [
        'Time to return results',
        'Memory and disk usage',
        'Basic keyword matching',
        'Multi-faceted queries',
        'Multiple constraints',
        'Understanding context',
        'Explainable results',
        'Ease of implementation',
        'Performance at scale',
        'Academic validation'
    ]
}

scorecard_df = pd.DataFrame(scorecard_data)

# Display scorecard
print("📋 COMPREHENSIVE SCORECARD (1-10 scale)")
print("=" * 80)
print(scorecard_df.to_string(index=False, 
                           formatters={'Dense Retrieval': '{:>3}'.format,
                                     'ColBERT': '{:>3}'.format}))

# Calculate totals
dense_total = sum(scorecard_df['Dense Retrieval'])
colbert_total = sum(scorecard_df['ColBERT'])

print(f"\n🏆 TOTAL SCORES:")
print(f"   Dense Retrieval: {dense_total}/100")
print(f"   ColBERT:         {colbert_total}/100")
print(f"   Winner: {'ColBERT' if colbert_total > dense_total else 'Dense' if dense_total > colbert_total else 'Tie'}")

# Visualize scorecard
plt.figure(figsize=(14, 10))

# Radar chart comparison
plt.subplot(2, 2, 1)
categories = scorecard_df['Criteria']
dense_scores = scorecard_df['Dense Retrieval'].values
colbert_scores = scorecard_df['ColBERT'].values

# Create angles for radar chart
angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False)
angles = np.concatenate((angles, [angles[0]]))

dense_scores = np.concatenate((dense_scores, [dense_scores[0]]))
colbert_scores = np.concatenate((colbert_scores, [colbert_scores[0]]))

ax = plt.subplot(2, 2, 1, projection='polar')
ax.plot(angles, dense_scores, 'o-', linewidth=2, label='Dense', color='lightblue')
ax.fill(angles, dense_scores, alpha=0.25, color='lightblue')
ax.plot(angles, colbert_scores, 'o-', linewidth=2, label='ColBERT', color='lightcoral')
ax.fill(angles, colbert_scores, alpha=0.25, color='lightcoral')

ax.set_xticks(angles[:-1])
ax.set_xticklabels([c.split()[0] for c in categories])  # Shortened labels
ax.set_ylim(0, 10)
ax.set_title('Performance Radar Chart')
ax.legend()

# Category-wise comparison
plt.subplot(2, 2, 2)
x = np.arange(len(categories))
width = 0.35

plt.bar(x - width/2, dense_scores[:-1], width, label='Dense', alpha=0.8, color='lightblue')
plt.bar(x + width/2, colbert_scores[:-1], width, label='ColBERT', alpha=0.8, color='lightcoral')

plt.xlabel('Criteria')
plt.ylabel('Score')
plt.title('Detailed Score Comparison')
plt.xticks(x, [c.split()[0] for c in categories], rotation=45, ha='right')
plt.legend()
plt.grid(True, alpha=0.3)

# Strengths and weaknesses analysis
plt.subplot(2, 2, 3)
dense_advantages = ['Speed', 'Storage', 'Simplicity', 'Scale']
colbert_advantages = ['Complex Queries', 'Multi-Constraint', 'Context', 'Interpretability']

plt.barh(range(len(dense_advantages)), [1]*len(dense_advantages), 
         alpha=0.7, color='lightblue', label='Dense Strengths')
plt.barh(range(len(dense_advantages), len(dense_advantages) + len(colbert_advantages)), 
         [1]*len(colbert_advantages), alpha=0.7, color='lightcoral', label='ColBERT Strengths')

all_advantages = dense_advantages + colbert_advantages
plt.yticks(range(len(all_advantages)), all_advantages)
plt.xlabel('Advantage Level')
plt.title('Key Strengths by Method')
plt.legend()

# Use case recommendations
plt.subplot(2, 2, 4)
plt.axis('off')

recommendations = """
🎯 WHEN TO USE EACH METHOD:

📦 Choose Dense Retrieval when:
• Speed is critical (real-time apps)
• Simple keyword searches
• Limited storage/compute
• Large scale deployment
• Quick prototype needed

🔍 Choose ColBERT when:
• Complex query understanding needed
• Multi-constraint searches
• Result interpretation important
• Quality over speed
• Rich contextual matching

💡 HYBRID APPROACH:
Use Dense for filtering + ColBERT for ranking!
"""

plt.text(0.05, 0.95, recommendations, transform=plt.gca().transAxes, 
         fontsize=10, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))

plt.tight_layout()
plt.show()

print("\n🏆 FINAL VERDICT:")
if colbert_total > dense_total:
    print(f"   ColBERT WINS overall ({colbert_total} vs {dense_total})")
    print(f"   Best for: Complex queries, interpretability, contextual understanding")
    print(f"   Trade-off: Slower speed, higher storage")
else:
    print(f"   Dense WINS overall ({dense_total} vs {colbert_total})")
    print(f"   Best for: Speed, simplicity, efficiency")
    print(f"   Trade-off: Less nuanced understanding")

print(f"\n🎯 For restaurant search specifically:")
print(f"   ColBERT excels at complex food/ambiance queries")
print(f"   Dense works well for simple restaurant name searches")
print(f"   Recommendation: ColBERT for better user experience!")

## 🎓 Key Takeaways for AI Tinkerers

### 🧠 What We Learned:

1. **Dense Retrieval** (Traditional RAG):
   - ⚡ **Fast**: Single vector comparison
   - 💾 **Efficient**: Low storage requirements
   - 🔍 **Simple**: Good for basic keyword matching
   - ⚠️ **Limited**: Struggles with complex, multi-faceted queries

2. **ColBERT** (Late Interaction):
   - 🎯 **Precise**: Token-level matching preserves nuance
   - 🧩 **Complex**: Handles multi-constraint queries excellently
   - 🔬 **Interpretable**: Can see which tokens matched
   - 📊 **Trade-offs**: Slower and uses more storage

### 🚀 The Future of RAG:

ColBERT represents the evolution from **early interaction** (dense) to **late interaction** (token-level). This shift enables:
- Better understanding of user intent
- More nuanced search results  
- Interpretable AI systems
- Foundation for even more advanced retrieval methods

### 💡 Practical Recommendations:

1. **Start with Dense** for prototypes and simple use cases
2. **Upgrade to ColBERT** when query complexity increases
3. **Consider Hybrid** approaches: Dense for speed + ColBERT for quality
4. **Choose based on your constraints**: Speed vs Quality vs Interpretability

---

*This concludes our comprehensive comparison! Both methods have their place in the RAG ecosystem. The key is choosing the right tool for your specific use case and constraints.*

**🎉 Ready to build your own advanced RAG system? You now have the knowledge to make informed decisions!**